# 👕 Virtual Try-On — Google Colab Setup

**Bilkul free. Koi payment nahi. Bas cells upar se neeche run karo.**

---

### Pehle Yeh Karo (Ek Baar):
1. Menu → **Runtime** → **Change runtime type**
2. Hardware accelerator = **T4 GPU** select karo
3. **Save** karo
4. Phir neeche cells run karo

---
| Cell | Kya karta hai | Time |
|------|--------------|------|
| 1 | GPU check | 5 sec |
| 2 | PyTorch + CUDA verify | 10 sec |
| 3 | Detectron2 install | 3-5 min |
| 4 | Project clone | 1-2 min |
| 5 | Requirements install | 3-5 min |
| 6 | DensePose model download | 1-2 min |
| 7 | App launch (public URL milega) | 15-20 min (model load) |

In [ ]:
# ── Cell 1: GPU Check ─────────────────────────────────────────────────────────
import torch

print('='*50)
print('GPU Available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU Name   :', torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('VRAM       :', vram_gb, 'GB')
    print('CUDA Ver   :', torch.version.cuda)
    print('PyTorch Ver:', torch.__version__)
    print('='*50)
    if vram_gb < 12:
        print('⚠️  WARNING: 12GB+ VRAM chahiye. T4 (15GB) use karo.')
    else:
        print('✅ GPU ready hai!')
else:
    print('='*50)
    print('❌ GPU nahi mila!')
    print('   Runtime > Change runtime type > T4 GPU select karo')
    raise SystemExit('GPU required — runtime type change karo phir restart karo')

In [ ]:
# ── Cell 2: Detectron2 Install (special — pip se direct nahi hota) ────────────
# Yeh automatically sahi version detect kar ke install karta hai

import torch
import subprocess

cuda_ver = torch.version.cuda.replace('.', '')[:3]   # e.g. '121'
torch_ver = torch.__version__.split('+')[0]           # e.g. '2.3.0'
torch_major_minor = 'torch' + '.'.join(torch_ver.split('.')[:2])  # e.g. 'torch2.3'

wheel_url = f'https://dl.fbaipublicfiles.com/detectron2/wheels/cu{cuda_ver}/{torch_major_minor}/index.html'

print(f'CUDA: {cuda_ver} | PyTorch: {torch_ver}')
print(f'Wheel URL: {wheel_url}')
print('Detectron2 install ho raha hai...')

# Pehle pre-built wheel try karo (fast)
result = subprocess.run(
    ['pip', 'install', 'detectron2', '-f', wheel_url, '-q'],
    capture_output=True, text=True
)

if result.returncode != 0:
    # Pre-built wheel nahi mila — source se build karo (slow but works)
    print('Pre-built wheel nahi mila. Source se build ho raha hai (5-10 min)...')
    subprocess.run(
        ['pip', 'install', 'git+https://github.com/facebookresearch/detectron2.git', '-q'],
        check=True
    )

# Verify
import detectron2
print(f'✅ Detectron2 {detectron2.__version__} install ho gaya!')

In [ ]:
# ── Cell 3: Project Setup ─────────────────────────────────────────────────────
# OPTION A: Google Drive se (zip upload karo Drive mein)
# OPTION B: GitHub se (agar repo hai)
# Neeche sirf jo option chahiye woh uncomment karo

import os, subprocess

PROJECT_DIR = '/content/demo-app'

# ═══════════════════════════════════════════════════════════════════
# OPTION A — Google Drive (Recommended agar GitHub nahi)
# ═══════════════════════════════════════════════════════════════════
# Step 1: drive.google.com pe demo-app.zip upload karo
# Step 2: Yeh cell run karo — Drive mount hoga, zip extract hoga

USE_DRIVE = True   # ← False karo agar GitHub use karna hai

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    # Apni zip file ka Drive path yahan set karo:
    ZIP_PATH = '/content/drive/MyDrive/demo-app.zip'  # ← yahi path hoga

    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            f'❌ Zip nahi mili: {ZIP_PATH}\n'
            '   Google Drive mein demo-app.zip upload karo phir dobara run karo.'
        )

    if os.path.isdir(PROJECT_DIR):
        print(f'✅ Project already hai: {PROJECT_DIR}')
    else:
        print('Zip extract ho rahi hai...')
        os.makedirs(PROJECT_DIR, exist_ok=True)
        subprocess.run(['unzip', '-q', ZIP_PATH, '-d', '/content/'], check=True)
        # Agar zip ke andar ek aur folder ho to yeh adjust karo:
        extracted = [d for d in os.listdir('/content/') if d.startswith('demo-app')]
        print(f'Extracted: {extracted}')
        print('✅ Extract complete!')

# ═══════════════════════════════════════════════════════════════════
# OPTION B — GitHub
# ═══════════════════════════════════════════════════════════════════
# GITHUB_URL = 'https://github.com/AAPKA_USERNAME/demo-app'
# if not USE_DRIVE:
#     subprocess.run(['git', 'clone', GITHUB_URL, PROJECT_DIR], check=True)
#     print('✅ Clone complete!')

# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)
print(f'\nWorking directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Requirements Install ─────────────────────────────────────────────
# detectron2 already install ho gaya Cell 2 mein, baaki sab yahan

print('Dependencies install ho rahi hain...')
print('(3-5 minute lag sakte hain)')

# HuggingFace pinned versions (pipeline ke liye zarori)
!pip install -q \
    diffusers==0.20.2 \
    transformers==4.31.0 \
    huggingface_hub==0.16.4 \
    accelerate==0.21.0 \
    gradio==3.41.2

# Image processing
!pip install -q \
    Pillow numpy scipy scikit-image opencv-python

# Model utilities
!pip install -q \
    einops safetensors onnxruntime omegaconf cloudpickle

# Detectron2 dependencies
!pip install -q \
    fvcore iopath pycocotools termcolor tabulate psutil

# Visualization + API server
!pip install -q \
    matplotlib tqdm basicsr \
    fastapi uvicorn[standard] python-multipart

print('✅ Sab dependencies install ho gayi!')

In [ ]:
# ── Cell 5: DensePose Model Download ─────────────────────────────────────────
import os, subprocess

# Make sure we're in project folder
os.chdir('/content/demo-app')

ckpt_dir  = '/content/demo-app/ckpt/densepose'
ckpt_file = os.path.join(ckpt_dir, 'model_final_162be9.pkl')

if os.path.exists(ckpt_file):
    size_mb = os.path.getsize(ckpt_file) / 1e6
    print(f'✅ DensePose model already hai ({size_mb:.0f} MB) — skip')
else:
    os.makedirs(ckpt_dir, exist_ok=True)
    print('DensePose model download ho raha hai (~250 MB)...')
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://dl.fbaipublicfiles.com/detectron2/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl',
        '-O', ckpt_file
    ], check=True)
    print(f'✅ Download complete: {ckpt_file}')

In [ ]:
# ── Cell 6: Final Verify (Run karne se pehle check) ──────────────────────────
import os

os.chdir('/content/demo-app')

checks = {
    'app.py'                                   : os.path.exists('app.py'),
    'api.py'                                   : os.path.exists('api.py'),
    'configs/densepose_rcnn_R_50_FPN_s1x.yaml' : os.path.exists('configs/densepose_rcnn_R_50_FPN_s1x.yaml'),
    'ckpt/densepose/model_final_162be9.pkl'    : os.path.exists('ckpt/densepose/model_final_162be9.pkl'),
    'src/tryon_pipeline.py'                    : os.path.exists('src/tryon_pipeline.py'),
    'preprocess/'                              : os.path.exists('preprocess'),
}

all_ok = True
print('=== Pre-flight Check ===')
for name, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status}  {name}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('✅ Sab kuch ready hai! Cell 7 run karo app launch karne ke liye.')
else:
    print('❌ Kuch cheez missing hai — upar ke cells dobara check karo.')

---
## App Launch Karo

**Cell 7** = Gradio Web UI (browser mein use karo)  
**Cell 8** = FastAPI (Postman / mobile app se test karo)  

Dono mein se ek hi chalao — dono ek saath nahi.

In [ ]:
# ── Cell 7: Gradio Web UI Launch ──────────────────────────────────────────────
# Public URL milega — kisi bhi browser/phone se open kar sako ge
# Pehli baar ~15-20 min (8GB model download)
# Doosri baar ~2-3 min

import os
os.chdir('/content/demo-app')

print('App launch ho rahi hai...')
print('Neeche "Running on public URL: https://..." wait karo')
print()

!python app.py

In [ ]:
# ── Cell 8: FastAPI + Public URL (Postman / Mobile App testing) ───────────────
# Yeh run karo agar Gradio ki jagah API endpoint chahiye

# pyngrok install karo (free ngrok tunnel)
!pip install -q pyngrok

import threading
import time
from pyngrok import ngrok

# API server background mein start karo
def start_api():
    os.system('python api.py')

thread = threading.Thread(target=start_api, daemon=True)
thread.start()

# Server ke start hone ka wait karo
print('API server start ho raha hai...')
time.sleep(8)

# Public tunnel banao
public_url = ngrok.connect(8000)
print()
print('='*60)
print(f'✅ API Public URL: {public_url}')
print('='*60)
print()
print('Postman mein yeh URL use karo:')
print(f'  POST {public_url}/tryon')
print(f'  POST {public_url}/tryon/both')
print(f'  GET  {public_url}/health')
print()
print('Docs: ' + str(public_url) + '/docs')

---
## Masail / Troubleshooting

| Masla | Hall |
|-------|------|
| `CUDA out of memory` | Runtime > Restart, phir Cell 7 directly run karo |
| `No module named detectron2` | Cell 2 dobara run karo |
| `model_final_162be9.pkl not found` | Cell 5 dobara run karo |
| URL nahi aa raha | 2-3 min wait karo, Cell 7 ki output scroll karo |
| Colab disconnect ho gaya | Sab cells fresh run karo (model cache rehta hai Drive mein nahi, phir download hoga) |

---
**Note:** Colab free tier mein session ~12 ghante chalta hai. Baad mein phir se run karna parega.